In [ ]:
# ============================================================
# Task 3: Energy Consumption Time Series Forecasting
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. LOAD & PARSE DATASET
# ============================================================
df = pd.read_csv(
    r"D:\User(D)\Download\test (1).csv",
    sep=';',
    parse_dates={'datetime': ['Date', 'Time']},
    dayfirst=True,
    na_values=['?'],
    low_memory=False
)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())
print("\nDate Range:", df['datetime'].min(), "→", df['datetime'].max())

print(df.head())

In [ ]:
# ============================================================
# 2. CLEAN & RESAMPLE
# ============================================================
# Set datetime as index
df.set_index('datetime', inplace=True)

# Convert all columns to numeric
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with missing values
df.dropna(inplace=True)

# Focus on Global Active Power (target variable)
target_col = 'Global_active_power'

# Resample to HOURLY averages (reduces noise, speeds up modeling)
df_hourly = df[[target_col]].resample('H').mean().dropna()

print(f"\nHourly resampled shape: {df_hourly.shape}")
print(df_hourly.head())



In [ ]:
# ============================================================
# 3. EXPLORATORY VISUALIZATION
# ============================================================
fig, axes = plt.subplots(3, 1, figsize=(16, 12))
fig.suptitle('Household Energy Consumption — EDA', fontsize=16, fontweight='bold')

# Plot 1: Full time series
df_hourly[target_col].plot(ax=axes[0], color='steelblue', linewidth=0.5)
axes[0].set_title('Hourly Global Active Power (Full Series)')
axes[0].set_ylabel('Power (kW)')
axes[0].set_xlabel('')

# Plot 2: One month sample
sample = df_hourly[target_col]['2007-01':'2007-02']
sample.plot(ax=axes[1], color='tomato', linewidth=0.8)
axes[1].set_title('Sample: January–February 2007')
axes[1].set_ylabel('Power (kW)')

# Plot 3: Rolling mean (smoothed trend)
rolling = df_hourly[target_col].rolling(window=24*7).mean()
df_hourly[target_col].plot(ax=axes[2], color='lightblue',
                            linewidth=0.4, label='Hourly')
rolling.plot(ax=axes[2], color='darkblue',
             linewidth=1.5, label='7-Day Rolling Mean')
axes[2].set_title('Trend with 7-Day Rolling Mean')
axes[2].set_ylabel('Power (kW)')
axes[2].legend()

plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# 4. FEATURE ENGINEERING (Time-based features)
# ============================================================
df_feat = df_hourly.copy()

df_feat['hour']        = df_feat.index.hour
df_feat['dayofweek']   = df_feat.index.dayofweek       # 0=Mon, 6=Sun
df_feat['month']       = df_feat.index.month
df_feat['year']        = df_feat.index.year
df_feat['quarter']     = df_feat.index.quarter
df_feat['is_weekend']  = (df_feat['dayofweek'] >= 5).astype(int)
df_feat['day_segment'] = pd.cut(df_feat['hour'],
                                 bins=[-1, 5, 11, 17, 23],
                                 labels=['Night','Morning','Afternoon','Evening'])
df_feat['day_segment'] = df_feat['day_segment'].astype(str)

# Lag features (past values as predictors)
df_feat['lag_1h']  = df_feat[target_col].shift(1)
df_feat['lag_24h'] = df_feat[target_col].shift(24)
df_feat['lag_7d']  = df_feat[target_col].shift(24*7)

# Rolling statistics
df_feat['roll_mean_24h'] = df_feat[target_col].shift(1).rolling(24).mean()
df_feat['roll_std_24h']  = df_feat[target_col].shift(1).rolling(24).std()

df_feat.dropna(inplace=True)

print("\nEngineered features shape:", df_feat.shape)
print(df_feat.head(3))



In [ ]:
# ============================================================
# 5. PATTERN ANALYSIS PLOTS
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Time-Based Usage Patterns', fontsize=14, fontweight='bold')

# Hour of day
hourly_avg = df_feat.groupby('hour')[target_col].mean()
hourly_avg.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Avg Power by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Avg Power (kW)')

# Day of week
day_avg = df_feat.groupby('dayofweek')[target_col].mean()
day_avg.index = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
day_avg.plot(kind='bar', ax=axes[1], color='tomato', edgecolor='black')
axes[1].set_title('Avg Power by Day of Week')
axes[1].set_xlabel('Day')
axes[1].tick_params(axis='x', rotation=45)

# Weekend vs Weekday
df_feat.groupby('is_weekend')[target_col].mean().plot(
    kind='bar', ax=axes[2],
    color=['steelblue','tomato'], edgecolor='black')
axes[2].set_title('Weekday vs Weekend Power')
axes[2].set_xticklabels(['Weekday','Weekend'], rotation=0)
axes[2].set_ylabel('Avg Power (kW)')

plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# 6. TRAIN/TEST SPLIT
# Use last 30 days as test set
# ============================================================
test_hours  = 24 * 30      # 30 days
train_data  = df_feat.iloc[:-test_hours]
test_data   = df_feat.iloc[-test_hours:]

print(f"\nTrain: {train_data.shape[0]} hours | Test: {test_data.shape[0]} hours")
print(f"Train period: {train_data.index[0]} → {train_data.index[-1]}")
print(f"Test  period: {test_data.index[0]}  → {test_data.index[-1]}")



In [ ]:
# ============================================================
# 7. MODEL 1 — ARIMA
# ============================================================
from statsmodels.tsa.arima.model import ARIMA

print("\nTraining ARIMA model...")

# Use daily resampled data for ARIMA (faster + less noise)
daily     = df_hourly.resample('D').mean().dropna()
train_d   = daily.iloc[:-30]
test_d    = daily.iloc[-30:]

arima_model  = ARIMA(train_d[target_col], order=(5, 1, 2))
arima_result = arima_model.fit()

arima_pred   = arima_result.forecast(steps=30)
arima_pred   = pd.Series(arima_pred.values, index=test_d.index)

arima_mae  = mean_absolute_error(test_d[target_col], arima_pred)
arima_rmse = np.sqrt(mean_squared_error(test_d[target_col], arima_pred))

print(f"ARIMA  →  MAE: {arima_mae:.4f}  |  RMSE: {arima_rmse:.4f}")



In [ ]:
# ============================================================
# 8. MODEL 2 — PROPHET
# ============================================================
from prophet import Prophet

print("\nTraining Prophet model...")

# Prophet requires columns named 'ds' and 'y'
prophet_train = train_d.reset_index().rename(
    columns={'datetime': 'ds', target_col: 'y'})
prophet_test  = test_d.reset_index().rename(
    columns={'datetime': 'ds', target_col: 'y'})

prophet_model = Prophet(
    daily_seasonality  = True,
    weekly_seasonality = True,
    yearly_seasonality = True,
    changepoint_prior_scale = 0.05
)
prophet_model.fit(prophet_train)

future         = prophet_model.make_future_dataframe(periods=30, freq='D')
prophet_fc     = prophet_model.predict(future)
prophet_pred   = prophet_fc[['ds','yhat']].tail(30)
prophet_pred.set_index('ds', inplace=True)

prophet_mae  = mean_absolute_error(test_d[target_col], prophet_pred['yhat'])
prophet_rmse = np.sqrt(mean_squared_error(test_d[target_col], prophet_pred['yhat']))

print(f"Prophet →  MAE: {prophet_mae:.4f}  |  RMSE: {prophet_rmse:.4f}")

# Prophet components plot
fig_p = prophet_model.plot_components(prophet_fc)
plt.suptitle('Prophet — Trend & Seasonality Components', fontsize=13)
plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# 9. MODEL 3 — XGBOOST
# ============================================================
from xgboost import XGBRegressor

print("\nTraining XGBoost model...")

feature_cols = ['hour','dayofweek','month','year','quarter',
                'is_weekend','lag_1h','lag_24h','lag_7d',
                'roll_mean_24h','roll_std_24h']

X_train_xgb = train_data[feature_cols]
y_train_xgb = train_data[target_col]
X_test_xgb  = test_data[feature_cols]
y_test_xgb  = test_data[target_col]

xgb_model = XGBRegressor(
    n_estimators      = 300,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    random_state      = 42,
    n_jobs            = -1
)
xgb_model.fit(X_train_xgb, y_train_xgb,
              eval_set=[(X_test_xgb, y_test_xgb)],
              verbose=False)

xgb_pred  = xgb_model.predict(X_test_xgb)
xgb_mae   = mean_absolute_error(y_test_xgb,  xgb_pred)
xgb_rmse  = np.sqrt(mean_squared_error(y_test_xgb, xgb_pred))

print(f"XGBoost →  MAE: {xgb_mae:.4f}  |  RMSE: {xgb_rmse:.4f}")



In [ ]:
# ============================================================
# 10. ACTUAL vs FORECASTED — ALL 3 MODELS
# ============================================================

# --- ARIMA & Prophet: daily plot ---
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Actual vs Forecasted Energy Consumption', fontsize=15, fontweight='bold')

# Top: ARIMA + Prophet (daily)
axes[0].plot(test_d.index, test_d[target_col],
             color='black', lw=2, label='Actual')
axes[0].plot(arima_pred.index, arima_pred,
             color='steelblue', lw=1.5, linestyle='--', label=f'ARIMA (RMSE={arima_rmse:.3f})')
axes[0].plot(prophet_pred.index, prophet_pred['yhat'],
             color='tomato', lw=1.5, linestyle='--', label=f'Prophet (RMSE={prophet_rmse:.3f})')
axes[0].set_title('ARIMA vs Prophet — Daily Forecast (30 Days)')
axes[0].set_ylabel('Global Active Power (kW)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Bottom: XGBoost (hourly — first 7 days for clarity)
plot_n = 24 * 7
axes[1].plot(test_data.index[:plot_n], y_test_xgb.values[:plot_n],
             color='black', lw=1.5, label='Actual')
axes[1].plot(test_data.index[:plot_n], xgb_pred[:plot_n],
             color='seagreen', lw=1.5, linestyle='--',
             label=f'XGBoost (RMSE={xgb_rmse:.3f})')
axes[1].set_title('XGBoost — Hourly Forecast (First 7 Days of Test)')
axes[1].set_ylabel('Global Active Power (kW)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# 11. XGBoost FEATURE IMPORTANCE
# ============================================================
xgb_feat_imp = pd.DataFrame({
    'Feature'   : feature_cols,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=xgb_feat_imp, x='Importance', y='Feature', palette='viridis')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# 12. MODEL COMPARISON SUMMARY
# ============================================================
results = pd.DataFrame({
    'Model': ['ARIMA', 'Prophet', 'XGBoost'],
    'MAE'  : [arima_mae,   prophet_mae,   xgb_mae],
    'RMSE' : [arima_rmse,  prophet_rmse,  xgb_rmse]
})
results['Rank_MAE']  = results['MAE'].rank().astype(int)
results['Rank_RMSE'] = results['RMSE'].rank().astype(int)

print("\n========== MODEL COMPARISON ==========")
print(results.to_string(index=False))
print(f"\nBest Model (RMSE): {results.loc[results['RMSE'].idxmin(), 'Model']}")

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

for ax, metric in zip(axes, ['MAE','RMSE']):
    sns.barplot(data=results, x='Model', y=metric,
                palette=['steelblue','tomato','seagreen'], ax=ax,
                edgecolor='black')
    ax.set_title(f'{metric} by Model (lower = better)')
    ax.set_ylabel(metric)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.4f}',
                    (p.get_x() + p.get_width()/2, p.get_height()),
                    ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
!pip install prophet xgboost statsmodels